# PCA - Tarefa 01: *HAR* com PCA

Vamos trabalhar com a base da demonstração feita em aula, mas vamos explorar um pouco melhor como é o desempenho da árvore variando o número de componentes principais.

In [9]:
import time
import pandas as pd

from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from time import perf_counter

In [6]:
base_path = "input/UCI HAR Dataset"

filename_features = f"{base_path}/features.txt"
filename_xtrain   = f"{base_path}/train/X_train.txt"
filename_ytrain   = f"{base_path}/train/y_train.txt"
filename_xtest    = f"{base_path}/test/X_test.txt"
filename_ytest    = f"{base_path}/test/y_test.txt"

print("Arquivos existem?")
print("features:", Path(filename_features).exists())
print("X_train :", Path(filename_xtrain).exists())
print("y_train :", Path(filename_ytrain).exists())
print("X_test  :", Path(filename_xtest).exists())
print("y_test  :", Path(filename_ytest).exists())

# Lendo features
df_features = pd.read_csv(
    filename_features,
    sep=r"\s+",
    header=None,
    names=["id_feature", "nome_feature"],
    engine="python"
)

features_raw = df_features["nome_feature"].tolist()

# Tornando nomes únicos (resolve duplicadas)
contagem = {}
features = []
for nome in features_raw:
    if nome in contagem:
        contagem[nome] += 1
        features.append(f"{nome}__{contagem[nome]}")
    else:
        contagem[nome] = 0
        features.append(nome)

dup_total = len(features_raw) - len(set(features_raw))
print(f"\nTotal de features: {len(features_raw)} | Duplicadas no arquivo: {dup_total}")

# Lendo X e y (treino e teste)
X_train = pd.read_csv(filename_xtrain, sep=r"\s+", header=None, names=features, engine="python")
y_train = pd.read_csv(filename_ytrain, header=None).iloc[:, 0]

X_test  = pd.read_csv(filename_xtest,  sep=r"\s+", header=None, names=features, engine="python")
y_test  = pd.read_csv(filename_ytest,  header=None).iloc[:, 0]

print("\nShapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train.head()


Arquivos existem?
features: True
X_train : True
y_train : True
X_test  : True
y_test  : True

Total de features: 561 | Duplicadas no arquivo: 84

Shapes:
X_train: (7352, 561)
y_train: (7352,)
X_test : (2947, 561)
y_test : (2947,)


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.074323,-0.298676,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,0.158075,-0.595051,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,0.414503,-0.390748,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,0.404573,-0.117290,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,0.087753,-0.351471,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892


## Árvore de decisão

Rode uma árvore de decisão com todas as variáveis, utilizando o ```ccp_alpha=0.001```. Avalie a acurácia nas bases de treinamento e teste. Avalie o tempo de processamento.

In [13]:
# Modelo (como pede o roteiro)
modelo_arvore = DecisionTreeClassifier(random_state=42, ccp_alpha=0.001)

# Tempo de treino
t0 = perf_counter()
modelo_arvore.fit(X_train, y_train)
t1 = perf_counter()

# Tempo de predição (treino + teste)
pred_train = modelo_arvore.predict(X_train)
pred_test = modelo_arvore.predict(X_test)
t2 = perf_counter()

# Métricas
acc_train = accuracy_score(y_train, pred_train)
acc_test = accuracy_score(y_test, pred_test)

tempo_treino = t1 - t0
tempo_pred = t2 - t1
tempo_total = t2 - t0

print("1) Árvore de decisão (todas as variáveis)")
print(f"Acurácia (treino): {acc_train:.4f}")
print(f"Acurácia (teste) : {acc_test:.4f}")
print(f"Tempo treino (s) : {tempo_treino:.4f}")
print(f"Tempo pred. (s)  : {tempo_pred:.4f}")
print(f"Tempo total (s)  : {tempo_total:.4f}")



1) Árvore de decisão (todas as variáveis)
Acurácia (treino): 0.9758
Acurácia (teste) : 0.8799
Tempo treino (s) : 3.0980
Tempo pred. (s)  : 0.0144
Tempo total (s)  : 3.1123


## Árvore com PCA

Faça uma análise de componentes principais das variáveis originais. Utilize apenas uma componente. Faça uma árvore de decisão com esta componente como variável explicativa.

- Avalie a acurácia nas bases de treinamento e teste
- Avalie o tempo de processamento

In [11]:
pipe_pca_1 = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=1, random_state=42)),
    ("arvore", DecisionTreeClassifier(random_state=42, ccp_alpha=0.001))
])

t0 = perf_counter()
pipe_pca_1.fit(X_train, y_train)
t1 = perf_counter()

pred_train_pca1 = pipe_pca_1.predict(X_train)
pred_test_pca1 = pipe_pca_1.predict(X_test)
t2 = perf_counter()

acc_train_pca1 = accuracy_score(y_train, pred_train_pca1)
acc_test_pca1 = accuracy_score(y_test, pred_test_pca1)

tempo_treino_pca1 = t1 - t0
tempo_pred_pca1 = t2 - t1
tempo_total_pca1 = t2 - t0

print("2) Árvore com PCA (1 componente)")
print(f"Acurácia (treino): {acc_train_pca1:.4f}")
print(f"Acurácia (teste) : {acc_test_pca1:.4f}")
print(f"Tempo treino (s) : {tempo_treino_pca1:.4f}")
print(f"Tempo pred. (s)  : {tempo_pred_pca1:.4f}")
print(f"Tempo total (s)  : {tempo_total_pca1:.4f}")


2) Árvore com PCA (1 componente)
Acurácia (treino): 0.4771
Acurácia (teste) : 0.4316
Tempo treino (s) : 0.1383
Tempo pred. (s)  : 0.0289
Tempo total (s)  : 0.1672


## Testando o número de componentes

Com base no código acima, teste a árvore de classificação com pelo menos as seguintes possibilidades de quantidades de componentes: ```[1, 2, 5, 10, 50]```. Avalie para cada uma delas:

- Acurácia nas bases de treino e teste
- Tempo de processamento


In [12]:
# fallback caso em algum momento você tenha X_train ao invés de x_train
Xtr = x_train if "x_train" in globals() else X_train
Xte = x_test  if "x_test"  in globals() else X_test
ytr = y_train
yte = y_test

componentes = [1, 2, 5, 10, 50]
resultados = []

for k in componentes:
    pipe = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=k, random_state=42)),
        ("arvore", DecisionTreeClassifier(random_state=42, ccp_alpha=0.001))
    ])
    
    t0 = perf_counter()
    pipe.fit(Xtr, ytr)
    t1 = perf_counter()
    
    pred_train_k = pipe.predict(Xtr)
    pred_test_k  = pipe.predict(Xte)
    t2 = perf_counter()
    
    acc_train_k = accuracy_score(ytr, pred_train_k)
    acc_test_k  = accuracy_score(yte, pred_test_k)
    
    tempo_treino_k = t1 - t0
    tempo_pred_k   = t2 - t1
    tempo_total_k  = t2 - t0
    
    resultados.append({
        "n_components": k,
        "acc_treino": acc_train_k,
        "acc_teste": acc_test_k,
        "tempo_treino_s": tempo_treino_k,
        "tempo_pred_s": tempo_pred_k,
        "tempo_total_s": tempo_total_k
    })

df_resultados = pd.DataFrame(resultados).sort_values("n_components").reset_index(drop=True)

print("3) Resultados - PCA + Árvore (variando n_components)")
print(df_resultados.to_string(index=False))
df_resultados.head()



3) Resultados - PCA + Árvore (variando n_components)
 n_components  acc_treino  acc_teste  tempo_treino_s  tempo_pred_s  tempo_total_s
            1    0.477149   0.431625        0.162075      0.028327       0.190402
            2    0.590180   0.546318        0.134444      0.033865       0.168309
            5    0.832699   0.770614        0.147957      0.032680       0.180637
           10    0.858678   0.767221        0.174737      0.040367       0.215104
           50    0.888194   0.775365        0.437194      0.037930       0.475125


,n_components,acc_treino,acc_teste,tempo_treino_s,tempo_pred_s,tempo_total_s
0,1,0.477149,0.431625,0.162075,0.028327,0.190402
1,2,0.590180,0.546318,0.134444,0.033865,0.168309
2,5,0.832699,0.770614,0.147957,0.032680,0.180637
3,10,0.858678,0.767221,0.174737,0.040367,0.215104
4,50,0.888194,0.775365,0.437194,0.037930,0.475125


## Conclua

- O que aconteceu com a acurácia?

    Comparando os resultados, a árvore de decisão sem PCA (utilizando todas as variáveis) apresentou a melhor acurácia no conjunto de teste (0,8799). Quando apliquei PCA com apenas 1 componente, a acurácia caiu de forma acentuada (teste 0,4316), indicando que uma única componente principal não consegue representar informação suficiente para separar bem as classes. Ao aumentar o número de componentes na Questão 3, a acurácia no teste melhorou progressivamente, saindo de 0,4316 (1 componente) para valores próximos de 0,77 com mais componentes (por exemplo, 0,7706 com 5 componentes, 0,7672 com 10 componentes e 0,7754 com 50 componentes). Mesmo assim, nenhuma configuração com PCA superou o desempenho do modelo sem PCA, que manteve a maior acurácia no teste.

- O que aconteceu com o tempo de processamento?

  O tempo total de processamento foi bem maior no modelo sem PCA (aproximadamente 3,1241 s). Ao aplicar PCA, o tempo total caiu bastante, ficando na faixa de aproximadamente 0,1683 s a 0,4751 s (dependendo do número de componentes). Observa-se que, conforme o número de componentes aumenta, o tempo tende a crescer, pois o pipeline passa a trabalhar com mais dimensões após a redução. Ainda assim, mesmo com 50 componentes, o tempo permaneceu menor do que o modelo sem PCA, mostrando que o PCA reduz o custo computacional, mas com a troca (trade-off) de uma perda de desempenho em acurácia quando comparado ao uso de todas as variáveis.